In [0]:
pip install statsmodels

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess   # for TS trend detection

# Functions to extract time series, perform train/test split, and detrend using orginial LOWESS detrending. (This motivates us to use a hybrid pandas/MLLib approach to achieve comparable results to our phase 2 and directly compare our ARIMA model's performance to MLLib. Pandas is used to extract the same deterministic trends we had in phase 2, while MLLib is used for future forecasting instead of ARIMA models from the statsmodels package.)

def country_indicator_view(df, country, indicator):
    df.createOrReplaceTempView("df")
    view = spark.sql(f"""
                     SELECT *
                     FROM df
                     WHERE `Indicator Name` = '{indicator}' AND `Country Name` = '{country}'
                     """
                     )
    return view

def get_ts(df, indicator, country):
    '''
    Function to extract single time series for country, indicator row from time series-formatted dataset

    Args:
        df - spark dataframe
        indicator - string matching target indicator
        country - string matching target country
    
    Returns:
        spark df formatted as time series (cols = year, indicator)
    '''
    target_row = country_indicator_view(df, country, indicator)

    if target_row.count() == 0:
        raise ValueError(f"No data found for {indicator} in {country}")

    cols = target_row.columns
    year_cols = [col for col in cols
                 if col not in ['Indicator Name', 'Country Name'] 
                 and col.isdigit()]
    
    target_row.createOrReplaceTempView("filtered")
    
    stack_parts = []
    for year in year_cols:
        stack_parts.append(f"'{year}', `{year}`")
    stack_expr = ", ".join(stack_parts)
    
    unpivoted = spark.sql(f"""
        SELECT 
            `Country Name`,
            `Indicator Name`,
            year,
            CAST(value AS DOUBLE) as value
        FROM filtered
        LATERAL VIEW STACK({len(year_cols)}, {stack_expr}) AS year, value
    """)
    
    spark_series = unpivoted.select(
        col("year").cast("date").alias("year"),
        col("value"),
        col("Country Name"),
        col("Indicator Name")
    ).orderBy("year").withColumn("year_index", row_number().over(Window.orderBy("year")) - 1)

    return spark_series

def train_test_split(spark_series, test_size):
    '''
    Function to split time series into training and test sets

    Args:
        series - spark series produced by get_ts
        test_size - float, proportion of series to use for training data 
    
    Returns:
        train_df - spark df containing training data
        test_df - spark df containing test data
    '''
    n = spark_series.count()
    n_test = int(n * test_size)
    n_train = n - n_test

    train_series = spark_series.limit(n_train)
    test_series = spark_series.exceptAll(train_series)
    
    return train_series, test_series

def plot_ts(train_series, indicator, country):
    '''
    Function to plot raw time series data

    Args:
        series - spark series produced by get_ts
        indicator - string, target indicator
        country - string, target country
        test_size - float, proportion of series to use for training data 
    
    Returns:
        no returns, plots time series
    '''
    # Converting to pandas for visualization purposes
    pd_df = train_series.select("year", "value").toPandas()
    pd_df.set_index('year', inplace=True)
    pd_series = pd_df['value']

    sns.set_theme(style="whitegrid") 

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=pd_series, x=pd_series.index, y=pd_series.value, color='blue')

    plt.xlabel('Year')
    plt.ylabel(indicator)
    plt.title(f'{country} {indicator} Over Time')

    plt.show()


def trend_detect(train_series, country, indicator, frac = 0.5):
    '''
    Function to determine trend component of time series using LOWESS (for annual data) 
    
    Args:
        series - spark time series df produced by get_ts
        frac - float, fraction of data to use for LOWESS smoothing
    
    Returns:
        trend - spark df of trend component
        residuals - spark df of residual component
    '''
    pd_df = train_series.select("year", "value").toPandas()
    pd_df.set_index('year', inplace=True)
    pd_series = pd_df['value']
    
    x = np.arange(len(pd_series))
    y = pd_series.values

    # Fitting LOWESS trend curve
    trend_values = lowess(y, x, frac=frac, return_sorted=False)
    trend = pd.Series(trend_values, index=pd_series.index, name='trend')
    
    # Calculating residuals (stochastic component of time series)
    residuals = pd_series - trend
    
    # Plotting time series with trend and residuals
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # Original time series with trend
    axes[0].plot(pd_series.index, pd_series, label='Original', color='blue', marker='o', alpha=0.7)
    axes[0].plot(trend.index, trend, label='Trend (LOWESS)', color='red', linewidth=2)
    axes[0].set_title(f'{country}: {indicator} - Training Series with Trend')
    axes[0].set_ylabel(indicator)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Detrended/Residual time series 
    axes[1].plot(residuals.index, residuals, color='green', marker='o')
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[1].set_title(f'{country}: {indicator} - Residuals (Detrended)')
    axes[1].set_ylabel('Residuals')
    axes[1].grid(True, alpha=0.3)
    
    # Residual histogram (for quick assessment, want to see normality)
    axes[2].hist(residuals.dropna(), bins=15, edgecolor='black', color='green', alpha=0.7)
    axes[2].axvline(x=0, color='red', linestyle='--')
    axes[2].set_title('Residual Distribution')
    axes[2].set_xlabel('Residual Value')
    axes[2].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nTrend and Residual Component Summary:")
    print(f"{'='*50}")
    print(f"Trend smoothing fraction: {frac}")
    print(f"Residual mean: {residuals.mean():.4f}")
    print(f"Residual std: {residuals.std():.4f}")
    print(f"Trend explains {100 * (1 - residuals.var() / pd_series.var()):.1f}% of variance")
    
    return trend, residuals

In [0]:
from pyspark.sql.functions import col, to_date, row_number, lit, when, isnan, isnull, lag, avg, stddev
import gc
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler, PolynomialExpansion
from pyspark.ml.regression import GBTRegressor, LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.linalg import Vectors
import numpy as np

#### Spark ML Lib Implementation of Time Series Forecasting with feature engineering ####

class TimeSeriesFeatureEngineering:
    @staticmethod
    def create_lag_features(train_series, max_lag=5):
        '''
        Function to create lag features for time series forecasting
        '''
        window_spec = Window.orderBy("year_index")
        
        for l in range(1, max_lag + 1):
            train_series = train_series.withColumn(f"lag_{l}", lag(col("value"), l).over(window_spec))
        
        return train_series
    
    @staticmethod
    def create_rolling_features(train_series, windows=[2, 3, 4, 5]):
        '''
        Function to create rolling avgerage and standard deviation features for time series forecasting
        '''
        window_spec = Window.orderBy("year_index").rowsBetween(-windows[-1], -1)
        
        for window in windows:
            # Rolling mean
            train_series = train_series.withColumn(f"rolling_mean_{window}", 
                            avg("value").over(Window.orderBy("year_index").rowsBetween(-window, -1)))
            # Rolling std
            train_series = train_series.withColumn(f"rolling_std_{window}", 
                            stddev("value").over(Window.orderBy("year_index").rowsBetween(-window, -1)))
            
        return train_series
    
    @staticmethod
    def prepare_features(train_series, max_lag=5, rolling_windows=[2, 3, 4, 5]):
        '''
        Function to prepare features lag, rolling average, and polynomial trend features for time series forecasting
        '''
        train_series = TimeSeriesFeatureEngineering.create_lag_features(train_series, max_lag)
        
        train_series = TimeSeriesFeatureEngineering.create_rolling_features(train_series, rolling_windows)

        return train_series

def build_pipeline(train_spark, features):
    '''
    Function to build machine learning pipeline for time series forecasting based on lags, moving avg/std dev, and polynomial features
    '''
    # Vector assembling and scaling
    assembler = VectorAssembler(inputCols=features, outputCol="features", handleInvalid="keep")
    
    scaler = StandardScaler(inputCol="features", outputCol="scaled_features",withStd=True, withMean=True)
    
    # model = LinearRegression(featuresCol="scaled_features", labelCol="residual")

    model = GBTRegressor(featuresCol="features", labelCol="residual",
    lossType="squared", seed=42) 
        
    pipeline = Pipeline(stages=[assembler, scaler, model])

    # Hyperparameter tuning

    # GBT param
    param_grid = ParamGridBuilder() \
        .addGrid(model.maxDepth, [3, 5]) \
        .addGrid(model.maxIter, [50]) \
        .addGrid(model.stepSize, [0.05, 0.1]) \
        .build()

    # Linreg param
    # param_grid = ParamGridBuilder() \
    # .addGrid(model.regParam, [0.0, 0.1, 1.0]) \
    # .build()
    
    # Evaluation of diff hyperparameters
    evaluator = RegressionEvaluator(labelCol="residual", predictionCol="prediction", metricName="rmse")
    
    # Cross validation step
    crossval = CrossValidator(estimator=pipeline, estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3, 
    seed=42,
    collectSubModels=False,
    parallelism=1)
    
    # Train model
    cv_model = crossval.fit(train_spark)
    
    # Get best model and parameters
    best_model = cv_model.bestModel
    best_params = cv_model.getEstimatorParamMaps()[np.argmin(cv_model.avgMetrics)]
    best_rmse = min(cv_model.avgMetrics)
    del cv_model

    print(f"\nBest Model Parameters:")
    for param, value in best_params.items():
        print(f"  {param.name}: {value}")
    print(f"Best Cross Validated RMSE: {best_rmse}")

    gc.collect()

    model_path = "/Volumes/workspace/default/raw/models/phase3_ts_model"

    dbutils.fs.mkdirs(model_path)

    best_model.write().overwrite().save(model_path)

    print("Model saved to:", model_path)
    
    return best_model

def forecast(df, indicator, country, test_size=0.1, max_lag=5):
    '''
    Function to forecast time series using Spark MLLib
    '''
    gc.collect()
    spark_series = get_ts(df, indicator, country)

    # Split train and test
    print("Splitting data...")
    train, test = train_test_split(spark_series, test_size=test_size)

    train_pd = train.select("year", "value", "year_index").toPandas()
    train_pd = train_pd.sort_values('year').reset_index(drop=True)
    train_pd.set_index('year', inplace=True)
    
    # Applying LOWESS trend detection on training data (requires pandas df)
    print("Applyng LOWESS trend detection on training data...")
    trend, residuals = trend_detect(train, country, indicator, frac=0.5)
    train_pd['trend'] = trend
    train_pd['residual'] = residuals

    # Converting back to Spark df for feature engineering and MLLib training
    train_spark = spark.createDataFrame(train_pd.reset_index())
    train_spark = train_spark.withColumnRenamed("index", "year").withColumn("year_index", row_number().over(Window.orderBy("year")) - 1)

    # Preparing training data features
    print("Feature engineering...")
    train_spark = TimeSeriesFeatureEngineering.prepare_features(train_spark, max_lag=max_lag)
    
    # Note that we don't train on the "value" column because it is accounted for by detrending step and other engineered features (lag, rolling avg, etc)
    features = ['rolling_mean_2', 'rolling_mean_3', 'rolling_mean_4', 'rolling_mean_5', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5']

    for feature in features:
        train_spark = train_spark.withColumn(feature, when(col(feature).isNull(), 0).when(isnan(col(feature)), 0).otherwise(col(feature)))

    # Training model
    print("Training model and performing 3-fold cross validation...")
    gc.collect()
    best_model = build_pipeline(train_spark, features)

    # Preparing test data features 
    test_pd = test.select("year", "value", "year_index").toPandas()
    test_spark = spark.createDataFrame(test_pd.reset_index())
    test_spark = test.withColumnRenamed("index", "year") \
                           .withColumn("year_index", 
                                      row_number().over(Window.orderBy("year")) - 1)
    
    # Applying LOWESS trend extrapolation for test period
    test_pd = test_pd.sort_values('year').reset_index(drop=True)
    test_pd.set_index('year', inplace=True)

    x_test = np.arange(len(train_pd), len(train_pd) + len(test_pd))
    
    # Extrapolating LOWESS trend using linear regression on recent trend points
    n_points = min(5, len(train_pd))
    recent_trend = train_pd['trend'].values[-n_points:]
    x_recent = np.arange(len(train_pd) - n_points, len(train_pd))
    
    slope, intercept = np.polyfit(x_recent, recent_trend, 1)
    trend_test = intercept + slope * x_test
    
    # Preparing test features in order to predict residuals  
    print("Performing feature engineering on test data...")
    test_spark = TimeSeriesFeatureEngineering.prepare_features(test_spark, max_lag=max_lag)

    for feature in features:
        test_spark = test_spark.withColumn(feature, when(col(feature).isNull(), 0).when(isnan(col(feature)), 0).otherwise(col(feature)))
    
    # Implementing MLLib model
    print("Predicting test residuals...")
    residual_predictions = best_model.transform(test_spark)
    
    preds = residual_predictions.select("prediction").collect()
    residual_forecast = np.array([p.prediction for p in preds])

    # Combining trend and residual forecasts
    test_predictions = trend_test[:len(residual_forecast)] + residual_forecast[:len(test_pd)]

    # Final metric evaluation
    actual_values = test_pd['value'].values[:len(test_predictions)]
    mae = np.mean(np.abs(actual_values - test_predictions))
    rmse = np.sqrt(np.mean((actual_values - test_predictions)**2))
    mape = np.mean(np.abs((actual_values - test_predictions) / (actual_values + 1e-8))) * 100

    # Plot results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: Test predictions vs actual
    axes[0, 0].plot(test_pd.index[:len(test_predictions)], actual_values, 
                   label='Actual', color='orange', marker='s')
    axes[0, 0].plot(test_pd.index[:len(test_predictions)], test_predictions, 
                   label='Forecast', color='red', linestyle='--', marker='x')
    axes[0, 0].set_title(f'Test Set Validation: {country} - {indicator}')
    axes[0, 0].set_ylabel(indicator)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Actual vs Predicted
    axes[0, 1].scatter(actual_values, test_predictions, alpha=0.7)
    axes[0, 1].plot([actual_values.min(), actual_values.max()], 
                   [actual_values.min(), actual_values.max()], 'r--')
    axes[0, 1].set_xlabel('Actual')
    axes[0, 1].set_ylabel('Predicted')
    axes[0, 1].set_title('Actual vs Predicted')
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: Residuals
    residuals_test = actual_values - test_predictions
    axes[1, 0].plot(test_pd.index[:len(test_predictions)], residuals_test, 
                   color='green', marker='o')
    axes[1, 0].axhline(y=0, color='black', linestyle='--')
    axes[1, 0].set_title('Forecast Residuals')
    axes[1, 0].set_ylabel('Residuals')
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Feature importance
    model_stage = best_model.stages[-1]
    importances = model_stage.featureImportances.toArray()
            
    # Get top 5 features
    indices = np.argsort(importances)[-5:][::-1]
    top_features = [features[i] for i in indices if i < len(features)]
    top_importances = [importances[i] for i in indices if i < len(features)]
            
    axes[1, 1].barh(top_features, top_importances)
    axes[1, 1].set_title('Top 5 Feature Importances')
    axes[1, 1].set_xlabel('Importance')
    
    plt.tight_layout()
    plt.show()
    
    metrics = {
        'mae': mae,
        'rmse': rmse,
        'mape': mape
    }
    
    return metrics

In [0]:
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.default.sparkml_temp
""")

import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/default/sparkml_temp'

In [0]:
data = spark.read.table("workspace.default.silver_merged_df")
data_wdi = spark.read.table("workspace.default.silver_wdi_df")

# Time Series Analysis 1 - Carbon Intensity of GDP in United States 
indicator1 = 'Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)'
country1 = 'United States'
ts = get_ts(data, indicator1, country1)

metrics1 = forecast(data, indicator1, country1)
print(metrics1)
